# Packages import

In [ ]:
import os
import re
import json
import requests
from dateutil.parser import isoparse
from datetime import timezone
from flask import Flask, render_template
from bs4 import BeautifulSoup

## News scraper

1. Provide URL address of article webpage

In [ ]:
article_code = "2026/06/cybersecurity-stars-awards-2026-winners.html"
url = f"https://thehackernews.com/{article_code}"

2. Send the request to provided URL address

In [ ]:
response = requests.get(url)
print(response.status_code)

3. If status code is OK, fetch article name

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

In [ ]:
article_name = page_dom.select_one("#app > div > h1 > a").get_text()
print(article_name)

4. If status code is OK, fetch article from requested webpage

In [ ]:
article = page_dom.select("#app > div")
print(type(article))
print(len(article))

5. For all fetched articles, parse them to extract relevant data

In [ ]:
whole_article = []
for ar in article:
    single_article = {
        'title': ar.select_one('h1 > a').text.strip(),
        'author': ar.select_one("div.postmeta > span.p-author > span:nth-child(2)").text.strip(),
        'pub_date': isoparse(ar.select_one('meta[itemprop="datePublished"]')["content"]).astimezone(timezone.utc).isoformat(),
        'mod_date': isoparse(ar.select_one('meta[itemprop="dateModified"]')["content"]).astimezone(timezone.utc).isoformat(),
        'tags': ar.select_one("div.postmeta > span.p-tags").text.strip(), 
        'body': ar.select_one("#articlebody").text.strip()
    }
    whole_article.append(single_article)

6. Save obtained article

In [ ]:
if not os.path.exists("./articles"):
    os.mkdir("./articles")

In [ ]:
safe_name = re.sub(r'[\\/*?:"<>|]', '', article_name).strip().replace(' ', '_')
fname = os.path.join("articles", f"{safe_name}.json")
json_text = json.dumps(whole_article, indent=4, ensure_ascii=False)
with open(fname, "w", encoding="utf-8") as jf:
    jf.write(json_text)

# Flask integration

I created a separate file (news_flask.py) so that Flask wouldn't parse the file again every time I started it

7. Create a Flask application

In [ ]:
app = Flask(__name__)

8. Load saved JSON data in Flask application

In [ ]:
def load_articles():
    with open(f"{safe_name}.json", "r", encoding="utf-8") as file:
        return json.load(file)

9. Create a route to display fetched articles

In [ ]:
@app.route("/")
def home():
    articles = load_articles()
    return render_template("index.html", articles=articles)

In [ ]:
@app.template_filter("pretty_date") # I wanted to prettify dates
def pretty_date(value):
    dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
    return dt.strftime("%d %B %Y, %H:%M")

10. Render article data using HTML template

I created index.html in templates folder and pasted html template with flask edits

11. Run Flask server and verify that scraped data is displayed correctly

In [ ]:
if __name__ == "__main__":
    app.run()